In [1]:
import pandas as pd
import sys
sys.path.append('..')
from src.url_features import extract_url_features
from src.html_features import extract_html_features
import time
import warnings
warnings.filterwarnings('ignore')

# Load Tranco and take top 5,000 domains to fetch HTML for
tranco = pd.read_csv('../dataset/tranco_top.csv', header=None, names=['rank', 'domain'])
tranco_fetch_sample = tranco.head(5000).copy()

print("Domains to fetch:", len(tranco_fetch_sample))
print(tranco_fetch_sample.head().to_string())

Domains to fetch: 5000
   rank            domain
0     1        google.com
1     2  gtld-servers.net
2     3    cloudflare.com
3     4       gstatic.com
4     5      facebook.com


In [2]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_one(domain):
    url_feats = extract_url_features(domain)
    html_feats = extract_html_features(domain)
    combined = {**url_feats, **html_feats}
    combined['url'] = domain
    combined['label'] = 0
    return combined

domains = tranco_fetch_sample['domain'].tolist()
tranco_html_list = []

print("Fetching with 30 parallel workers...")
with ThreadPoolExecutor(max_workers=30) as executor:
    futures = {executor.submit(fetch_one, d): d for d in domains}
    done = 0
    for future in as_completed(futures):
        tranco_html_list.append(future.result())
        done += 1
        if done % 200 == 0:
            print(f"Progress: {done}/{len(domains)}")

df_tranco_combined = pd.DataFrame(tranco_html_list)
print("\nDone! Shape:", df_tranco_combined.shape)
print("Successfully fetched:", df_tranco_combined['fetch_status'].sum(), "out of", len(df_tranco_combined))

Fetching with 30 parallel workers...


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Progress: 200/5000


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Progress: 400/5000


...

Progress: 4400/5000
Progress: 4600/5000


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Progress: 4800/5000


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Progress: 5000/5000

Done! Shape: (5000, 43)
Successfully fetched: 2557 out of 5000


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils import resample
import joblib

# Load original combined fetched dataset
df_orig_combined = pd.read_csv('../dataset/combined_features.csv')
print("Original combined shape:", df_orig_combined.shape)
print("Columns match:", list(df_orig_combined.columns) == list(df_tranco_combined.columns))

# Combine original + Tranco benign
df_hybrid_aug = pd.concat([df_orig_combined, df_tranco_combined], ignore_index=True)

# Keep only successfully fetched
df_fetched = df_hybrid_aug[df_hybrid_aug['fetch_status'] == 1]
print("\nTotal fetched samples:", len(df_fetched))
print("Label distribution:\n", df_fetched['label'].value_counts())

# Balance the two classes
benign = df_fetched[df_fetched['label'] == 0]
malicious = df_fetched[df_fetched['label'] == 1]
n = min(len(benign), len(malicious))
benign_bal = resample(benign, replace=False, n_samples=n, random_state=42)
malicious_bal = resample(malicious, replace=False, n_samples=n, random_state=42)
df_bal = pd.concat([benign_bal, malicious_bal]).sample(frac=1, random_state=42).reset_index(drop=True)
print("\nBalanced samples:", len(df_bal))
print(df_bal['label'].value_counts())

# Train
X = df_bal.drop(['url', 'label', 'fetch_status'], axis=1)
y = df_bal['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("\nTraining augmented hybrid model...")
rf_hybrid_aug = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_hybrid_aug.fit(X_train, y_train)

y_pred = rf_hybrid_aug.predict(X_test)
print("\n=== Augmented Hybrid Model Performance ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")

# Test google.com on the new hybrid
print("\n=== Testing google.com ===")
url_f = extract_url_features('google.com')
html_f = extract_html_features('google.com')
if html_f['fetch_status'] == 1:
    combined = {**url_f, **html_f}
    combined.pop('fetch_status', None)
    test_df = pd.DataFrame([combined])
    pred = rf_hybrid_aug.predict(test_df)[0]
    prob = rf_hybrid_aug.predict_proba(test_df)[0]
    print("Prediction:", "Malicious" if pred == 1 else "Benign")
    print("Probability [benign, malicious]:", prob)
else:
    print("Could not fetch google.com right now")

Original combined shape: (10000, 43)
Columns match: True

Total fetched samples: 8964
Label distribution:
 label
0    6275
1    2689
Name: count, dtype: int64

Balanced samples: 5378
label
1    2689
0    2689
Name: count, dtype: int64

Training augmented hybrid model...

=== Augmented Hybrid Model Performance ===
Accuracy:  0.7928
Precision: 0.7656
Recall:    0.8439
F1 Score:  0.8028

=== Testing google.com ===
Prediction: Benign
Probability [benign, malicious]: [1. 0.]


In [4]:
import shap
import pickle

# Save the augmented hybrid model
joblib.dump(rf_hybrid_aug, '../model/hybrid_model.pkl')
print("Augmented hybrid model saved!")

# Regenerate the hybrid SHAP explainer to match
print("Regenerating hybrid SHAP explainer...")
explainer_hybrid = shap.TreeExplainer(rf_hybrid_aug)
with open('../model/shap_explainer_hybrid.pkl', 'wb') as f:
    pickle.dump(explainer_hybrid, f)
print("Hybrid SHAP explainer regenerated and saved!")

Augmented hybrid model saved!
Regenerating hybrid SHAP explainer...
Hybrid SHAP explainer regenerated and saved!
